"""
M6 Plotly 互動儀表板 & Capstone — 課後作業
===========================================
情境：從原始資料到互動式儀表板，完成完整的資料分析 pipeline。

資料路徑：
  - datasets/ecommerce/orders_raw.csv（原始髒資料）
  - datasets/ecommerce/customers.csv
  - datasets/ecommerce/products.csv
"""

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = 'plotly_white'  # 乾淨白底



In [2]:
df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)
print('資料形狀:', df.shape)

資料形狀: (188, 14)


# ============================================================
# 🟢 送分題（每題 10 分，共 30 分）
# ============================================================


In [3]:
df

,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level,product_name,category,unit_price,stock_qty
0,5064,2022,1026,4.0,2025-03-26,8600.0,Victor Lin,North,2023-02-27,Gold,Dumbbell 5kg,Sports,2150,51
1,5023,2026,1021,5.0,2025-01-05,1355.0,Zoe Huang,South,2023-05-16,Platinum,Throw Pillow,Home,271,150
2,5123,2013,1013,2.0,2025-09-11,3538.0,Mia Huang,North,2023-07-17,Platinum,Cotton T-Shirt,Clothing,1769,174
3,5118,2005,1028,1.0,2025-05-22,1618.0,Emma Liu,West,2023-05-18,Bronze,Water Bottle,Sports,1618,186
4,5161,2017,1019,3.0,2025-08-20,1846.0,Quinn Chen,East,2023-08-11,Silver,Coffee Mug,Home,1846,274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,5094,2026,1019,3.0,2025-02-13,5538.0,Zoe Huang,South,2023-05-16,Platinum,Coffee Mug,Home,1846,274
184,5041,2014,1001,5.0,2025-10-03,8135.0,Nick Huang,West,2023-09-28,Gold,Wireless Mouse,Electronics,1627,12
185,5157,2005,1026,5.0,2025-01-02,10750.0,Emma Liu,West,2023-05-18,Bronze,Dumbbell 5kg,Sports,2150,51
186,5134,2015,1012,5.0,2025-06-03,9580.0,Olivia Huang,North,2023-12-15,Bronze,Clean Code,Books,1916,81


In [4]:
def green_plotly_bar():
    """
    用 Plotly Express 畫出每個商品類別 (category) 的總營收長條圖
    資料來源：orders_enriched.csv
    回傳 plotly Figure 物件
    提示：px.bar()
    """
    # TODO: 你的程式碼

category_rev = df.groupby('category', as_index=False)['amount'].sum().sort_values('amount', ascending=False)
fig = px.bar(category_rev, x='category', y='amount', text='amount',
             color='category', title='Revenue by category')
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()


In [5]:
def green_plotly_line():
    """
    用 Plotly Express 畫出月營收趨勢折線圖
    資料來源：orders_enriched.csv
    回傳 plotly Figure 物件
    提示：先 groupby 月份算總營收，再 px.line()
    """
    # TODO: 你的程式碼
    

df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False)['amount'].sum()

fig = px.line(monthly, x='month', y='amount', markers=True,
                    title='Monthly Revenue Trend')
fig.update_layout(height=400)
fig.show()



In [6]:
def green_plotly_pie():
    """
    用 Plotly Express 畫出 VIP 等級 (vip_level) 的訂單數佔比圓餅圖
    資料來源：orders_enriched.csv
    回傳 plotly Figure 物件
    提示：px.pie()
    """
    # TODO: 你的程式碼
vip_qty = df.groupby('vip_level', as_index=False)['qty'].sum()
fig = px.pie(vip_qty, names='vip_level', values='qty',
             title='VIP Level qty', hole=0.4)  # hole=0.4 變成 donut
fig.update_layout(height=400)
fig.show()


# ============================================================
# 🟡 核心題（每題 15 分，共 45 分）
# ============================================================


In [7]:
def yellow_clean_and_merge(orders_raw, customers, products):
    """
    完整 ETL：從髒資料到合併完成的 DataFrame
    1. 讀取 orders_raw.csv 並清理（欄位名稱、金額、日期、缺值、去重）
    2. 合併 customers.csv 和 products.csv
    回傳：合併後的 DataFrame
    """
    # TODO: 你的程式碼
def clean_orders(orders_raw):
    df = pd.read_csv('../datasets/ecommerce/orders_raw.csv')
    df.columns = df.columns.str.strip().str.lower()
    df['amount'] = (
            df['amount'].astype(str)
            .str.replace('$','', regex=False)
            .str.replace(',','', regex=False)
            .astype(float)
        )
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df = df.dropna(subset=['order_date'])
    df['qty'] = df['qty'].fillna(df['qty'].median())
    df = df.drop_duplicates()
    return df

raw_orders = pd.read_csv('../datasets/ecommerce/orders_raw.csv')
orders = clean_orders(raw_orders)
print(f'清理完成：{orders.shape[0]} 筆訂單')
orders.head(3)

customers = pd.read_csv('../datasets/ecommerce/customers.csv')
products  = pd.read_csv('../datasets/ecommerce/products.csv')

enriched = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print(f'合併後形狀: {enriched.shape}')
print(f'欄位數: {len(enriched.columns)}')

print(enriched)

清理完成：188 筆訂單
合併後形狀: (188, 14)
欄位數: 14
     order_id  customer_id  product_id  qty order_date   amount customer_name  \
0        5064         2022        1026  4.0 2025-03-26   8600.0    Victor Lin   
1        5023         2026        1021  5.0 2025-01-05   1355.0     Zoe Huang   
2        5123         2013        1013  2.0 2025-09-11   3538.0     Mia Huang   
3        5118         2005        1028  1.0 2025-05-22   1618.0      Emma Liu   
4        5161         2017        1019  3.0 2025-08-20   1846.0    Quinn Chen   
..        ...          ...         ...  ...        ...      ...           ...   
183      5094         2026        1019  3.0 2025-02-13   5538.0     Zoe Huang   
184      5041         2014        1001  5.0 2025-10-03   8135.0    Nick Huang   
185      5157         2005        1026  5.0 2025-01-02  10750.0      Emma Liu   
186      5134         2015        1012  5.0 2025-06-03   9580.0  Olivia Huang   
187      5135         2010        1007  4.0 2025-09-05   2436.0      Ja

In [8]:
def yellow_kpi_summary(df):
    """
    計算 4 個核心 KPI，回傳 dict：
    {
        "total_revenue": float,       # 總營收
        "order_count": int,           # 訂單數
        "active_customers": int,      # 不重複客戶數
        "avg_order_value": float,     # 平均客單價
    }
    """
    # TODO: 你的程式碼

enriched['month'] = enriched['order_date'].dt.to_period('M').astype(str)
kpis = {
    '總營收':     enriched['amount'].sum(),
    '總訂單數':   len(enriched),
    '活躍顧客數': enriched['customer_id'].nunique(),
    '客單價':     enriched['amount'].sum() / len(enriched),
}
for k, v in kpis.items():
    print(f'{k}: {v:>12,.0f}')

總營收:      686,388
總訂單數:          188
活躍顧客數:           26
客單價:        3,651


In [12]:
def yellow_plotly_scatter(df):
    """
    用 Plotly Express 畫互動散佈圖：
    - X：商品單價 (unit_price)
    - Y：訂單金額 (amount)
    - 顏色：商品類別 (category)
    - hover 顯示：商品名稱 (product_name)
    回傳 plotly Figure 物件
    提示：px.scatter(hover_data=['product_name'])
    """
    # TODO: 你的程式碼
fig = px.scatter(df, x='unit_price', y='amount',
                 color='category', hover_data=['product_name'],
                 title='Unit Price')
fig.update_layout(height=450)
fig.show()

# ============================================================
# 🔴 挑戰題（25 分）
# ============================================================

In [13]:
def red_dashboard():
    """
    Capstone：完整的互動式儀表板

    流程：
    1. 清理 orders_raw.csv + 合併三張表
    2. 建立 2×2 subplot dashboard（用 plotly make_subplots）：
       - 左上：月營收趨勢 (line)
       - 右上：Top 10 商品營收 (bar)
       - 左下：各地區營收 (bar)
       - 右下：類別營收佔比 (pie/donut)
    3. 設定整體標題

    回傳 plotly Figure 物件
    提示：from plotly.subplots import make_subplots
    """
    # TODO: 你的程式碼
monthly     = enriched.groupby('month', as_index=False)['amount'].sum()
top_prod    = (enriched.groupby('product_name', as_index=False)['amount']
               .sum().sort_values('amount', ascending=False).head(10))
region_rev  = enriched.groupby('region', as_index=False)['amount'].sum()
cat_rev     = enriched.groupby('category', as_index=False)['amount'].sum()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend',
                    'Top 10 Products',
                    'Revenue by Region',
                    'Category Share'),
    specs=[[{'type':'xy'}, {'type':'xy'}],
           [{'type':'xy'}, {'type':'domain'}]],
)

fig.add_trace(go.Line(x=monthly['month'], y=monthly['amount'],
                         mode='lines+markers', name='Monthly'), row=1, col=1)
fig.add_trace(go.Bar(x=top_prod['product_name'], y=top_prod['amount'],
                     name='Top Products'), row=1, col=2)
fig.add_trace(go.Bar(x=region_rev['region'], y=region_rev['amount'],
                     name='Region'), row=2, col=1)
fig.add_trace(go.Pie(labels=cat_rev['category'], values=cat_rev['amount'],
                     name='Category', hole=0.4), row=2, col=2)

fig.update_layout(
    title_text='<b>E-Commerce Sales Dashboard — 2025</b>',
    height=750, showlegend=False,
)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.show()


c:\Users\highm\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotly\graph_objs\_deprecations.py:378: DeprecationWarning: plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.

  warnings.warn(
